In [ ]:
!pip install PyMuPDF
import fitz

def read_pdf(file_path):
    doc = fitz.open(file_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

pdf_text = read_pdf('/content/Lab_4_AutoEncoder.pdf')
print(pdf_text[:1000]) # Preview first 1000 characters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 64.5 MB/s eta 0:00:00
 
 
 
 
1 
 
THỰC HÀNH 5 : AUTOENCODER 
Sau bài thực hành này, sinh viên có khả năng: 
- 
Xây dựng được kiến trúc Convolutional Neural Network (CNN) 
- 
Cài đặt mô hình Convolutional Neural Network bằng tensorflow 
- 
Cài đặt ứng dụng thực tế sử dụng Convolutional Neural Network 
1. GIỚI THIỆU  
Autoencoder nhận một ảnh chưa gán nhãn và rút các đặc trưng quan trọng của một đối tượng (ảnh). 
Đây là phương pháp học không giám sát.  
Thường Autoencoder có 2 thành phần: 
- 
Encoder: Nhận dữ liệu đầu vào và nén trong không gian hidden. Gọi x là input data, E là hàm 
encoder thì biểu diễn đầu ra không gian ẩn là s = E(x) 
- 
Decoder: nhận không gian ẩn s từ encoder và tái dựng lại dữ liệu ban đầu. Gọi D là hàm 
decoder, o là đầu ra thì o = D(s) 
- 
Về mặt toán học, toàn bộ quá trình autoencoder sẽ được viết lại như sau: o = D(E(x)) 
 
2. CÀI ĐẶT MÔ HÌNH AUTOENCODER 
Phần này sẽ trình bày cài đặt mô hình autoencode

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, losses
from tensorflow.keras.models import Model
from flask import Flask, render_template, request, send_from_directory
from google.colab.output import eval_js
import cv2

# 1. Khởi tạo thư mục cho Flask
if not os.path.exists('static/uploads'):
    os.makedirs('static/uploads')
if not os.path.exists('templates'):
    os.makedirs('templates')

# 2. Chuẩn bị dữ liệu MNIST
(x_train, _), (x_test, _) = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype('float32') / 255.
x_test = x_test.astype('float32') / 255.

# 3. Định nghĩa Autoencoder
class Autoencoder(Model):
    def __init__(self, latent_dim=64):
        super(Autoencoder, self).__init__()
        self.latent_dim = latent_dim
        self.encoder = tf.keras.Sequential([
            layers.Flatten(),
            layers.Dense(latent_dim, activation='relu'),
        ])
        self.decoder = tf.keras.Sequential([
            layers.Dense(784, activation='sigmoid'),
            layers.Reshape((28, 28))
        ])

    def call(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

autoencoder = Autoencoder(64)
autoencoder.compile(optimizer='adam', loss=losses.MeanSquaredError())
autoencoder.fit(x_train, x_train, epochs=5, shuffle=True, validation_data=(x_test, x_test))

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 17s 7ms/step - loss: 0.0240 - val_loss: 0.0093
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 0.0070 - val_loss: 0.0054
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 0.0051 - val_loss: 0.0046
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 0.0046 - val_loss: 0.0043
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - loss: 0.0044 - val_loss: 0.0042


In [ ]:
html_content = """
<!DOCTYPE html>
<html>
<head>
    <title>Autoencoder Fashion MNIST</title>
    <link href='https://cdn.jsdelivr.net/npm/bootstrap@5.1.3/dist/css/bootstrap.min.css' rel='stylesheet'>
    <style>
        body { background: #f4f7f6; padding: 50px; }
        .card { border-radius: 15px; box-shadow: 0 10px 20px rgba(0,0,0,0.1); }
        .img-box { border: 1px solid #ddd; padding: 10px; border-radius: 10px; background: #fff; min-height: 100px; }
    </style>
</head>
<body>
    <div class='container'>
        <div class='card p-4 mx-auto' style='max-width: 700px;'>
            <h2 class='text-center mb-4'>Phục hồi ảnh Fashion MNIST</h2>
            <form action='/' method='post' enctype='multipart/form-data'>
                <div class='mb-3'>
                    <input class='form-control' type='file' name='file' required>
                </div>
                <button type='submit' class='btn btn-success w-100'>Xử lý ngay</button>
            </form>

            {% if original %}
            <div class='row mt-5 text-center'>
                <div class='col-md-6'>
                    <h5>Ảnh gốc</h5>
                    <div class='img-box'><img src='/{{ original }}' class='img-fluid'></div>
                </div>
                <div class='col-md-6'>
                    <h5>Ảnh phục hồi</h5>
                    <div class='img-box'><img src='/{{ reconstructed }}' class='img-fluid'></div>
                </div>
            </div>
            {% endif %}
        </div>
    </div>
</body>
</html>
"""

with open('templates/index.html', 'w') as f:
    f.write(html_content)

In [ ]:
app = Flask(__name__, static_url_path='/static')

@app.route('/', methods=['GET', 'POST'])
def index():
    if request.method == 'POST':
        file = request.files['file']
        if file:
            filename = file.filename
            filepath = os.path.join('static/uploads', filename)
            file.save(filepath)

            # Đọc và tiền xử lý ảnh
            img = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)
            img_resized = cv2.resize(img, (28, 28)) / 255.0
            input_batch = np.expand_dims(img_resized, axis=0)

            # Dự đoán qua mô hình
            reconstructed_img = autoencoder(input_batch).numpy()[0] * 255
            out_name = 'res_' + filename
            out_path = os.path.join('static/uploads', out_name)
            cv2.imwrite(out_path, reconstructed_img.astype(np.uint8))

            return render_template('index.html', original=filepath, reconstructed=out_path)
    return render_template('index.html', original=None)

@app.route('/static/uploads/<path:filename>')
def serve_static(filename):
    return send_from_directory('static/uploads', filename)

print("Truy cập vào link bên dưới để sử dụng App:")
print(eval_js("google.colab.kernel.proxyPort(5000)"))

if __name__ == '__main__':
    app.run(port=5000)

In [ ]:
html_content = """
<!DOCTYPE html>
<html>
<head>
    <title>Autoencoder Fashion MNIST</title>
    <link href='https://cdn.jsdelivr.net/npm/bootstrap@5.1.3/dist/css/bootstrap.min.css' rel='stylesheet'>
    <style>
        body { background: #f4f7f6; padding: 50px; }
        .card { border-radius: 15px; box-shadow: 0 10px 20px rgba(0,0,0,0.1); }
        .img-box { border: 1px solid #ddd; padding: 10px; border-radius: 10px; background: #fff; min-height: 100px; }
    </style>
</head>
<body>
    <div class='container'>
        <div class='card p-4 mx-auto' style='max-width: 700px;'>
            <h2 class='text-center mb-4'>Phục hồi ảnh Fashion MNIST</h2>
            <form action='/' method='post' enctype='multipart/form-data'>
                <div class='mb-3'>
                    <input class='form-control' type='file' name='file' required>
                </div>
                <button type='submit' class='btn btn-success w-100'>Xử lý ngay</button>
            </form>

            {% if original %}
            <div class='row mt-5 text-center'>
                <div class='col-md-6'>
                    <h5>Ảnh gốc</h5>
                    <div class='img-box'><img src='/{{ original }}' class='img-fluid'></div>
                </div>
                <div class='col-md-6'>
                    <h5>Ảnh phục hồi</h5>
                    <div class='img-box'><img src='/{{ reconstructed }}' class='img-fluid'></div>
                </div>
            </div>
            {% endif %}
        </div>
    </div>
</body>
</html>
"""

with open('templates/index.html', 'w') as f:
    f.write(html_content)

In [ ]:
app = Flask(__name__, static_url_path='/static')

@app.route('/', methods=['GET', 'POST'])
def index():
    if request.method == 'POST':
        file = request.files['file']
        if file:
            filename = file.filename
            filepath = os.path.join('static/uploads', filename)
            file.save(filepath)

            # Đọc và tiền xử lý ảnh (chuyển về xám và resize 28x28)
            img = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)
            img_resized = cv2.resize(img, (28, 28)) / 255.0
            input_batch = np.expand_dims(img_resized, axis=0)

            # Dự đoán qua mô hình
            reconstructed_img = autoencoder(input_batch).numpy()[0] * 255
            out_name = 'res_' + filename
            out_path = os.path.join('static/uploads', out_name)
            cv2.imwrite(out_path, reconstructed_img.astype(np.uint8))

            return render_template('index.html', original=filepath, reconstructed=out_path)
    return render_template('index.html', original=None)

@app.route('/static/uploads/<path:filename>')
def serve_static(filename):
    return send_from_directory('static/uploads', filename)

print("Truy cập vào link bên dưới để sử dụng App:")
print(eval_js("google.colab.kernel.proxyPort(5000)"))

if __name__ == '__main__':
    app.run(port=5000)

Truy cập vào link bên dưới để sử dụng App:
https://5000-m-s-kkb-usc1c0-245g6pnjep6o8-c.us-central1-0.prod.colab.dev
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [06/Jun/2026 03:44:00] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [06/Jun/2026 03:44:00] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [06/Jun/2026 03:54:12] "GET / HTTP/1.1" 200 -
